# 00 Setup
**Lab 3B, Credit Watch.** Check your environment, copy the seed workspace into `agent/`, and run the tests.

Everything runs offline on the **fixture**: four fictional BDCs, two fictional client mandates, recorded EDGAR and release feeds, and a deterministic stand-in for a model. Your Case Study 2B panel replaces the fixture panel when you switch to real data (Case Study 3B, Section 4).

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
import importlib.util, platform
print("Python", platform.python_version(), "| pandas", pd.__version__)
for m, why in [("yaml", "required"), ("mcp", "Part 1: pip install 'mcp[cli]>=2.1,<3'"), ("airflow", "Part 5: run it in Docker")]:
    print(f"  {m:8s}", "ok" if importlib.util.find_spec(m) else f"not installed ({why})")
import sqlite3
try:
    sqlite3.connect(":memory:").execute("CREATE VIRTUAL TABLE t USING fts5(x)")
    print("  sqlite   FTS5 available (Part 4 memory search)")
except sqlite3.OperationalError:
    print("  sqlite   FTS5 MISSING: use a Python build with FTS5 (python.org, conda or Homebrew), or sqlite-vec")

In [ ]:
agent = ROOT / P["agent"]["workspace"]
if agent.exists():
    print("agent/ already exists: left untouched (it is your team's working copy)")
else:
    workspace.fresh_workspace(ROOT / P["agent"]["seed"], agent)
    print("agent/ created from the seed:", sorted(p.name for p in agent.iterdir()))

In [ ]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "tests" / "run_tests.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-700:])
assert r.returncode == 0, "the tests should pass on the fixture before you start"

In [ ]:
from creditwatch import check
res = check.run_checks(ROOT)
print(f"Case Study 3B self-check: {sum(ok for _, ok, _ in res)}/{len(res)} pass. On the bare kit, the FAILs are your to-do list:")
for name, ok, _ in res:
    print(f"  {'PASS' if ok else 'todo'}  {name}")

**Your turn.** Rename the agent in `agent/SOUL.md`, and read `agent/AGENTS.md` and `agent/HEARTBEAT.md`: Case Study 3B, Part 0 grades them. Run the self-check whenever you like, from your `cs3b/` folder: `PYTHONPATH=src python -m creditwatch.check`.